In [1]:
#%%
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path

from monoculture.analysis.setup import (
    ACS_TASKS,
    TABLESHIFT_TASKS,
    LLM_MODELS,
    prompt_connectors,
    prompt_styles,
)
from monoculture.analysis.utils import (
    model_to_key,
    is_instruction_tuned,
    load_json,
    find_files,
    parse_results_dict,
    create_result_df,
)
import pandas as pd

TASKS = ACS_TASKS + TABLESHIFT_TASKS
RESULTS_ROOT_DIR = Path("./results/")
subfolders = [
    "tableshift/0-bullet-is",
    "tableshift/10-reuse-bullet-is",
    "folktexts/0-bullet-is",
    "folktexts/0-bullet-colon",
    "folktexts/0-bullet-equal",
    "folktexts/0-text-is",
    "folktexts/10-reuse-bullet-is",
]  # , 'few-shot']

#### smaller version for one prompting style

In [12]:
model_folders = [dir+'/'+dir.replace('model-','')+f'_task-{task}' for task in TASKS for folder in subfolders for dir in os.listdir(RESULTS_ROOT_DIR/folder) if dir.startswith('model')]
cond_finished = lambda files: files and any([f.endswith('.json') for f in files]) and any([f.endswith('predictions.csv') for f in files])
finished_benches = [path for fold in model_folders for sub in subfolders for path,_,file_list in os.walk(RESULTS_ROOT_DIR/sub/fold) if cond_finished(file_list)]
finished_benches[:3], len(finished_benches)

(['results/folktexts/0-bullet-is/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3535356034',
  'results/folktexts/0-bullet-colon/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3440500242',
  'results/folktexts/0-bullet-equal/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-4235015672'],
 1717)

In [13]:
model_folders

['model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome',
 'model-google--gemma-2-27b-it/google--gemma-2-27b-it_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-70B/meta-llama--Meta-Llama-3-70B_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-8B/meta-llama--Meta-Llama-3-8B_task-ACSIncome',
 'model-google--gemma-2-9b/google--gemma-2-9b_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-8B-Instruct/meta-llama--Meta-Llama-3-8B-Instruct_task-ACSIncome',
 'model-mistralai--Mistral-7B-v0.1/mistralai--Mistral-7B-v0.1_task-ACSIncome',
 'model-mistralai--Mistral-7B-Instruct-v0.2/mistralai--Mistral-7B-Instruct-v0.2_task-ACSIncome',
 'model-google--gemma-2b/google--gemma-2b_task-ACSIncome',
 'model-mistralai--Mixtral-8x22B-v0.1/mistralai--Mixtral-8x22B-v0.1_task-ACSIncome',
 'model-mistralai--Mixtral-8x7B-v0.1/mistralai--Mixtral-8x7B-v0.1_task-ACSIncome',
 'model-meta-llama--Meta-Llama-3-70B-Instruct/meta-llama--Meta-Llama-3-70B-Instruct_task-ACSIncome',
 'model-01-ai--Yi-34B/01-ai--Yi-34

In [14]:
finished_benches

['results/folktexts/0-bullet-is/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3535356034',
 'results/folktexts/0-bullet-colon/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-3440500242',
 'results/folktexts/0-bullet-equal/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-4235015672',
 'results/folktexts/0-text-is/model-google--gemma-2-27b/google--gemma-2-27b_task-ACSIncome/google--gemma-2-27b_bench-54056871',
 'results/folktexts/0-bullet-is/model-google--gemma-2-27b-it/google--gemma-2-27b-it_task-ACSIncome/google--gemma-2-27b-it_bench-1373954821',
 'results/folktexts/0-bullet-colon/model-google--gemma-2-27b-it/google--gemma-2-27b-it_task-ACSIncome/google--gemma-2-27b-it_bench-3421353447',
 'results/folktexts/0-bullet-equal/model-google--gemma-2-27b-it/google--gemma-2-27b-it_task-ACSIncome/google--gemma-2-27b-it_bench-715593136',
 'results/folktexts/0-text-is/model-google

In [15]:
model_task_pairs = []
for folder_name in finished_benches:
    (model, task) = folder_name.split('/')[-2].split('_task-')
    model = model.replace('model-', '')
    model_task_pairs.append((model, task))
unique_model_task_pairs = list(set(model_task_pairs))
unique_model_task_pairs[0]

('google--gemma-2-9b', 'BRFSS_Blood_Pressure')

In [16]:
unique_model_task_pairs

[('google--gemma-2-9b', 'BRFSS_Blood_Pressure'),
 ('google--gemma-2-27b-it', 'ACSEmployment'),
 ('mistralai--Mixtral-8x22B-v0.1', 'ACSEmployment'),
 ('meta-llama--Meta-Llama-3.1-8B', 'ACSIncome'),
 ('Qwen--Qwen2-7B-Instruct', 'ACSPublicCoverage'),
 ('mistralai--Mixtral-8x22B-v0.1', 'ACSIncome'),
 ('meta-llama--Meta-Llama-3-70B', 'ACSTravelTime'),
 ('google--gemma-2-9b', 'ACSTravelTime'),
 ('meta-llama--Meta-Llama-3-8B-Instruct', 'BRFSS_Diabetes'),
 ('google--gemma-2-9b-it', 'ACSIncome'),
 ('meta-llama--Meta-Llama-3.2-1B', 'ACSPublicCoverage'),
 ('mistralai--Mixtral-8x22B-Instruct-v0.1', 'ACSTravelTime'),
 ('meta-llama--Meta-Llama-3-8B', 'BRFSS_Diabetes'),
 ('mistralai--Mixtral-8x7B-v0.1', 'BRFSS_Diabetes'),
 ('meta-llama--Meta-Llama-3-8B-Instruct', 'BRFSS_Blood_Pressure'),
 ('allenai--OLMo-1B-0724-hf', 'ACSTravelTime'),
 ('meta-llama--Meta-Llama-3-8B', 'BRFSS_Blood_Pressure'),
 ('mistralai--Mixtral-8x7B-v0.1', 'BRFSS_Blood_Pressure'),
 ('google--gemma-1.1-2b-it', 'BRFSS_Blood_Pressure'

In [22]:
show_available = False
show_unavailable = True
for task in TASKS:
    print(task)
    for model in LLM_MODELS:
        if (model_to_key(model), task) in unique_model_task_pairs:
            if show_available:
                print(f"- {model_to_key(model)}")
        else:
            if show_unavailable:
                print(f'x {model_to_key(model)}')
    print()

ACSIncome

ACSEmployment
x Qwen--Qwen2-72B-Instruct

ACSTravelTime
x Qwen--Qwen2-72B-Instruct

ACSPublicCoverage

BRFSS_Diabetes
x meta-llama--Meta-Llama-3.1-8B
x meta-llama--Meta-Llama-3.1-8B-Instruct
x meta-llama--Meta-Llama-3.1-70B
x meta-llama--Meta-Llama-3.1-70B-Instruct
x meta-llama--Meta-Llama-3.2-1B
x meta-llama--Meta-Llama-3.2-1B-Instruct
x meta-llama--Meta-Llama-3.2-3B
x meta-llama--Meta-Llama-3.2-3B-Instruct
x 01-ai--Yi-6B-Chat
x Qwen--Qwen2-1.5B
x Qwen--Qwen2-1.5B-Instruct
x Qwen--Qwen2-7B
x Qwen--Qwen2-7B-Instruct
x Qwen--Qwen2-72B
x Qwen--Qwen2-72B-Instruct
x mlfoundations--tabula-8b
x allenai--OLMo-1B-0724-hf
x allenai--OLMo-1B-hf
x allenai--OLMo-7B-0724-hf
x allenai--OLMo-7B-hf
x allenai--OLMo-7B-Instruct-hf
x allenai--OLMo-2-1124-7B
x allenai--OLMo-2-1124-7B-Instruct

BRFSS_Blood_Pressure
x meta-llama--Meta-Llama-3.1-8B
x meta-llama--Meta-Llama-3.1-8B-Instruct
x meta-llama--Meta-Llama-3.1-70B
x meta-llama--Meta-Llama-3.1-70B-Instruct
x meta-llama--Meta-Llama-3.2-1B
x m

### Check availability for different prompting styles

In [3]:
SAVE_DIR = RESULTS_ROOT_DIR
save_file_path = SAVE_DIR/'overview_results_by_prompt_style.csv'

In [4]:
load_df = False
df = pd.read_csv(save_file_path) if load_df else create_result_df(RESULTS_ROOT_DIR, subfolders=subfolders, tasks=TASKS, save_path=save_file_path)

(1052, 10)
Saving dataframe to results/overview_results_by_prompt_style.csv


In [5]:
print(df.shape)
df[(df['num_shots']==0) & (df['task']=='ACSIncome') & (df["threshold_fitted"] == 0)].head()

(1052, 10)


,task,model,is_inst,threshold_fitted,bench_hash,num_shots,prompt_style,prompt_connector,eval_results_path,predictions_path
0,ACSIncome,meta-llama--Meta-Llama-3.2-1B-Instruct,1,0,4087643733,0,bullet,is,results/folktexts/0-bullet-is/model-meta-llama...,results/folktexts/0-bullet-is/model-meta-llama...
3,ACSIncome,allenai--OLMo-2-1124-7B-Instruct,1,0,2229201787,0,bullet,is,results/folktexts/0-bullet-is/model-allenai--O...,results/folktexts/0-bullet-is/model-allenai--O...
5,ACSIncome,google--gemma-2-27b,0,0,1719521954,0,bullet,is,results/folktexts/0-bullet-is/model-google--ge...,results/folktexts/0-bullet-is/model-google--ge...
7,ACSIncome,google--gemma-2-27b-it,1,0,1665971878,0,bullet,is,results/folktexts/0-bullet-is/model-google--ge...,results/folktexts/0-bullet-is/model-google--ge...
8,ACSIncome,meta-llama--Meta-Llama-3-70B,0,0,758164074,0,bullet,is,results/folktexts/0-bullet-is/model-meta-llama...,results/folktexts/0-bullet-is/model-meta-llama...


In [14]:
show_available = False
show_unavailable = True
fitted_treshold = False

for task_name in ACS_TASKS[:2]+TABLESHIFT_TASKS: # ACS_TASKS[:2]:  # TABLESHIFT_TASKS+ACS_TASKS:
    print(task_name)
    for num_shots in [10]:  ## only check zero-shot for now
        for format in ["bullet"]:  # ['bullet', 'text']:
            for con in ["is"]:  # ['is', 'colon', 'equal']:
                if format == "text" and con != "is":
                    continue
                else:
                    print(f"  {num_shots} {format} {con}")
                for m in LLM_MODELS:
                    num_entries = df[
                        (df["task"] == task_name)
                        & (df["model"] == model_to_key(m))
                        & (df["prompt_style"] == format)
                        & (df["prompt_connector"] == con)
                        & (df["num_shots"] == num_shots)
                        & (df["threshold_fitted"] == int(fitted_treshold))
                    ].shape[0]
                    if show_available:
                        if num_entries == 1:
                            # print(f"\t- {m} ")
                            print(f"--model={m}", end =" ")
                        elif num_entries > 1:
                            print(
                                f"\t- {m} -- Found multiple models with given characteristics."
                            )
                    if show_unavailable and num_entries == 0:
                        print(f"\tx {m}") #, end =" ")
                        # print(f"--model={m}", end =" ")
                print()

ACSIncome
  10 bullet is
	x Qwen/Qwen2.5-72B
	x Qwen/Qwen2.5-72B-Instruct

ACSEmployment
  10 bullet is
	x meta-llama/Meta-Llama-3.1-70B
	x meta-llama/Meta-Llama-3.1-70B-Instruct
	x meta-llama/Meta-Llama-3.3-70B-Instruct
	x mistralai/Mistral-Small-24B-Base-2501
	x mistralai/Mistral-Small-24B-Instruct-2501
	x Qwen/Qwen2.5-72B
	x Qwen/Qwen2.5-72B-Instruct

BRFSS_Diabetes
  10 bullet is
	x google/gemma-2-9b
	x google/gemma-2-9b-it
	x google/gemma-2-27b
	x google/gemma-2-27b-it
	x meta-llama/Meta-Llama-3.1-70B
	x meta-llama/Meta-Llama-3.1-70B-Instruct
	x meta-llama/Meta-Llama-3.3-70B-Instruct
	x mistralai/Mistral-Small-24B-Instruct-2501
	x Qwen/Qwen2.5-72B
	x Qwen/Qwen2.5-72B-Instruct

BRFSS_Blood_Pressure
  10 bullet is
	x google/gemma-2-9b
	x google/gemma-2-9b-it
	x google/gemma-2-27b
	x google/gemma-2-27b-it
	x meta-llama/Meta-Llama-3.1-70B
	x meta-llama/Meta-Llama-3.1-70B-Instruct
	x meta-llama/Meta-Llama-3.3-70B-Instruct
	x Qwen/Qwen2.5-72B
	x Qwen/Qwen2.5-72B-Instruct

